In [8]:
# ============================================================
# 🚀 PIPELINE FINAL: METABOLISMO + CLÍNICO + KM (VERSIÓN 2026 v2)
# ============================================================
# Mejoras:
#   ✅ Agrupación clínica en macro-categorías biológicas reales
#   ✅ Agrupación metabólica en 5 bloques funcionales
#   ✅ Top 12 por p_adj + efecto (log2FC)
#   ✅ Análisis integrado clínica ↔ metabolismo
#   ✅ Estructura de reporte narrativa científica
# ============================================================

# 1. CARGA DE LIBRERÍAS
required_libs <- c("dplyr", "ggplot2", "reshape2", "ggpubr", "survival",
                   "survminer", "readxl", "purrr", "tidyr", "grid", "gridExtra",
                   "scales")
invisible(lapply(required_libs, library, character.only = TRUE))

# ===============================
# 2. CONFIGURACIÓN DE RUTAS
# ===============================
paths <- list(
  metabolic    = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Metabolic data analysis/ML models using metabolic data/resultados_TumorPhenotype_UMAP_metrics_actualizado_sinl2/Merged_TumorPhenotype_UMAP_AllData.csv",
  clinicaldata = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Clinical data analysis/ML models using clinical data/Results_clustering_UMAP/DATA_MASTER_CLUSTERS_Y_CLINICA.csv",
  div          = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_divergentes_para_estudio_pyn.csv",
  core         = "/Users/eduardoruiz/Documents/GitHub/Precision-Oncology-for-Breast-Cancer-Diagnosis/src/Clustering and data analysis PYTHON/Cluster_correlations/pacientes_core_correlacion_Paretoynormas.csv"
)

group_colors <- c("Core" = "#0072B2", "Divergent" = "#D55E00")

# ─── TEMA GLOBAL ────────────────────────────────────────────────────────────
base_theme <- theme_pubr() +
  theme(
    title         = element_text(size = 20, face = "bold"),
    axis.title    = element_text(size = 17),
    axis.text     = element_text(size = 15),
    legend.text   = element_text(size = 15),
    legend.title  = element_text(size = 16, face = "bold"),
    strip.text    = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 15, color = "grey50")
  )

# ===============================
# 3. VARIABLES METABÓLICAS
# ===============================
SOLS <- c("FBA", "pFBA", "L1w", "L2", "L2w")

expand_sols <- function(roots, sols = SOLS) {
  as.vector(outer(roots, sols, paste, sep = "_"))
}

SCALAR_ROOTS <- c(
  "CU", "EA",
  "WarburgIndex",
  "ATPConsumption", "ATPProduction",
  "RedoxIndex", "MFI", "AnabolismScore",
  "NADPHdemand", "TCA_completeness",
  "LipidSat", "LipidUnsat", "LipidPL",
  "GlnDependence"
)
SCALAR_VARS <- expand_sols(SCALAR_ROOTS)

ONCOMET_NAMES <- c("Lactate", "Succinate", "AlphaKG")
ONCOMET_VARS  <- as.vector(outer(
  paste0("Oncomet_", ONCOMET_NAMES), SOLS, paste, sep = "_"
))

ALL_METAB_VARS_STATIC <- c(SCALAR_VARS, ONCOMET_VARS)

# ===============================
# 4. PREPROCESAMIENTO Y CRUCE
# ===============================
clean_names <- function(x) toupper(gsub("\\.", "-", trimws(as.character(x))))

df_metab <- read.csv(paths$metabolic,    stringsAsFactors = FALSE)
df_clin  <- read.csv(paths$clinicaldata, stringsAsFactors = FALSE)
df_div   <- read.csv(paths$div,          stringsAsFactors = FALSE)
df_core  <- read.csv(paths$core,         stringsAsFactors = FALSE)

df_metab$ModelName <- clean_names(df_metab$ModelName)
df_clin$ModelName  <- clean_names(df_clin$ModelName)
div_names          <- clean_names(df_div$ModelName)
core_names         <- clean_names(df_core$ModelName)

# ─── Detectar columnas SA_* ──────────────────────────────────────────────────
sa_cols_csv <- grep(
  paste0("^SA_.+_(", paste(SOLS, collapse = "|"), ")$"),
  colnames(df_metab), value = TRUE
)
cat("\n🔍 Subsistemas SA detectados en CSV:", length(sa_cols_csv), "\n")
if (length(sa_cols_csv) > 0) {
  subsys_unique <- unique(sub(paste0("_(", paste(SOLS, collapse="|"), ")$"), "", sa_cols_csv))
  cat("   Subsistemas únicos:", length(subsys_unique), "\n")
  cat(paste0("   • ", head(subsys_unique, 10), collapse = "\n"), "\n")
  if (length(subsys_unique) > 10) cat("   ...", length(subsys_unique) - 10, "más\n")
}

ALL_METAB_VARS <- c(ALL_METAB_VARS_STATIC, sa_cols_csv)

# ─── Detectar colisiones entre grupos ────────────────────────────────────────
overlap <- intersect(div_names, core_names)
if (length(overlap) > 0) {
  warning(paste0("⚠️  ", length(overlap), " pacientes en ambos grupos (se excluirán): ",
                 paste(head(overlap, 5), collapse = ", ")))
}

analysis <- df_metab %>%
  mutate(Group = case_when(
    ModelName %in% overlap    ~ NA_character_,
    ModelName %in% div_names  ~ "Divergent",
    ModelName %in% core_names ~ "Core",
    TRUE                      ~ "Other"
  )) %>%
  filter(Group %in% c("Core", "Divergent"))

# ─── Verificar variables metabólicas disponibles ─────────────────────────────
metab_available <- intersect(ALL_METAB_VARS, colnames(analysis))
metab_missing   <- setdiff(ALL_METAB_VARS,  colnames(analysis))

cat("\n📊 Variables metabólicas encontradas:", length(metab_available), "/", length(ALL_METAB_VARS))
if (length(metab_missing) > 0) {
  cat("\n⚠️  Variables NO encontradas (", length(metab_missing), "):\n")
  cat(paste0("   • ", head(metab_missing, 20), collapse = "\n"), "\n")
  if (length(metab_missing) > 20) cat("   ...", length(metab_missing) - 20, "más\n")
}

# ─── Join clínico ────────────────────────────────────────────────────────────
clinical_cols <- c(
  "ModelName",
  "ajcc_pathologic_stage.diagnoses",
  "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",
  "ajcc_pathologic_m.diagnoses",
  "tumor_grade.diagnoses",
  "morphology.diagnoses",
  "primary_diagnosis.diagnoses",
  "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses",
  "prior_treatment.diagnoses",
  "sample_type.samples",
  "tissue_type.samples",
  "tumor_descriptor.samples",
  "specimen_type.samples",
  "is_ffpe.samples",
  "oct_embedded.samples",
  "age_at_diagnosis.diagnoses",
  "alcohol_history.exposures",
  "Menopausal.Status",
  "Cancer.Type",
  "ER", "PR", "HER2",
  "Subtype",
  "Genetic.Ancestry",
  "Prior_Treatment_Flag",
  "Metastasis_Flag",
  "ER_PR_HER2_Combo",
  "ER_PR_HER2_Combo_encoded",
  "Subtype_encoded"
)

clin_subset   <- df_clin %>% select(any_of(clinical_cols))
cols_from_clin <- setdiff(colnames(clin_subset), "ModelName")

analysis <- analysis %>%
  select(-any_of(cols_from_clin)) %>%
  left_join(clin_subset, by = "ModelName") %>%
  rename_with(~ gsub("Menopausal\\.Status", "Menopausal Status", .x)) %>%
  rename_with(~ gsub("Cancer\\.Type",       "Cancer Type",       .x)) %>%
  rename_with(~ gsub("Genetic\\.Ancestry",  "Genetic Ancestry",  .x))

cat("\n✅ Pacientes Core:", sum(analysis$Group == "Core"),
    " | Divergent:", sum(analysis$Group == "Divergent"), "\n")

# ===============================
# 5A. GRUPOS CLÍNICOS (MACRO-CATEGORÍAS BIOLÓGICAS)
# ===============================
# 🧬 A. Biología Tumoral — fenotipo molecular y agresividad
CLIN_GROUP_A <- intersect(c(
  "ER", "PR", "HER2",
  "Subtype",
  "ER_PR_HER2_Combo",
  "tumor_grade.diagnoses",
  "morphology.diagnoses",
  "primary_diagnosis.diagnoses"
), colnames(analysis))

# 📊 B. Estadio y Carga Tumoral — progresión estructural
CLIN_GROUP_B <- intersect(c(
  "ajcc_pathologic_stage.diagnoses",
  "ajcc_pathologic_t.diagnoses",
  "ajcc_pathologic_n.diagnoses",
  "ajcc_pathologic_m.diagnoses",
  "Metastasis_Flag"
), colnames(analysis))

# 💊 C. Historia Terapéutica — divergencia metabólica adaptativa
CLIN_GROUP_C <- intersect(c(
  "prior_treatment.diagnoses",
  "treatment_type.treatments.diagnoses",
  "treatment_or_therapy.treatments.diagnoses",
  "Prior_Treatment_Flag"
), colnames(analysis))

# 🧍‍♀️ D. Factores del Paciente — moduladores sistémicos
CLIN_GROUP_D <- intersect(c(
  "age_at_diagnosis.diagnoses",
  "Menopausal Status",
  "Genetic Ancestry",
  "alcohol_history.exposures"
), colnames(analysis))

CLIN_GROUPS <- list(
  "🧬 A. Biología Tumoral"       = CLIN_GROUP_A,
  "📊 B. Estadio y Carga Tumoral" = CLIN_GROUP_B,
  "💊 C. Historia Terapéutica"    = CLIN_GROUP_C,
  "🧍 D. Factores del Paciente"   = CLIN_GROUP_D
)
CLIN_GROUPS <- Filter(function(v) length(v) > 0, CLIN_GROUPS)

cat("\n📂 Grupos clínicos activos:", length(CLIN_GROUPS), "\n")
for (g in names(CLIN_GROUPS)) {
  cat("   •", g, ":", length(CLIN_GROUPS[[g]]), "variables\n")
}

# ===============================
# 5B. GRUPOS METABÓLICOS FUNCIONALES (5 BLOQUES)
# ===============================
make_group <- function(roots, available, sols = SOLS) {
  intersect(expand_sols(roots, sols), available)
}

METAB_GROUPS <- list(
  # 🔥 1. Energía y Bioenergética
  "🔥 1. Energía y Bioenergética" = make_group(
    c("ATPConsumption", "ATPProduction", "WarburgIndex", "MFI", "CU", "EA"),
    metab_available
  ),
  # 🔴 2. Estado Redox y Anabolismo
  "🔴 2. Estado Redox y Anabolismo" = make_group(
    c("RedoxIndex", "NADPHdemand", "AnabolismScore", "TCA_completeness"),
    metab_available
  ),
  # 🧈 3. Metabolismo Lipídico
  "🧈 3. Metabolismo Lipídico" = make_group(
    c("LipidSat", "LipidUnsat", "LipidPL"),
    metab_available
  ),
  # 🧪 4. Dependencias Metabólicas Específicas
  "🧪 4. Dependencias Metabólicas Específicas" = c(
    make_group("GlnDependence", metab_available),
    intersect(ONCOMET_VARS, metab_available)
  ),
  # 🧩 5. Reprogramación Sistémica (SA_*)
  "🧩 5. Reprogramación Sistémica (SA_*)" = sa_cols_csv[sa_cols_csv %in% metab_available]
)
METAB_GROUPS <- Filter(function(v) length(v) > 0, METAB_GROUPS)

cat("\n📂 Grupos metabólicos funcionales activos:", length(METAB_GROUPS), "\n")
for (g in names(METAB_GROUPS)) {
  cat("   •", g, ":", length(METAB_GROUPS[[g]]), "variables\n")
}

# ===============================
# 6. ANÁLISIS ESTADÍSTICO (FDR)
# ===============================
run_stats <- function(data, columns) {
  results <- map_df(columns, function(col) {
    sub_data <- data %>%
      select(Group, value = !!sym(col)) %>%
      filter(is.finite(value), !is.na(value))

    group_sizes <- table(sub_data$Group)
    if (length(group_sizes) < 2 || any(group_sizes < 3)) return(NULL)

    test <- wilcox.test(value ~ Group, data = sub_data, exact = FALSE)

    meds <- sub_data %>%
      group_by(Group) %>%
      summarise(m = median(value), .groups = "drop")

    data.frame(
      Feature  = col,
      Med_Core = meds$m[meds$Group == "Core"],
      Med_Div  = meds$m[meds$Group == "Divergent"],
      p_val    = test$p.value
    )
  })

  if (nrow(results) == 0) return(results)

  results %>%
    mutate(
      p_adj       = p.adjust(p_val, method = "fdr"),
      Fold_Change = Med_Div / ifelse(Med_Core == 0, NA, Med_Core)
    ) %>%
    arrange(p_val)
}

metab_res <- run_stats(analysis, metab_available)

cat("\n📈 Variables metabólicas significativas (p_adj < 0.05):",
    sum(metab_res$p_adj < 0.05, na.rm = TRUE), "/", nrow(metab_res), "\n")

write.csv(metab_res, "Estadisticas_Metabolicas_CoreVsDivergent.csv", row.names = FALSE)
cat("✅ Tabla estadística exportada: Estadisticas_Metabolicas_CoreVsDivergent.csv\n")

# ─── Resumen por grupo metabólico ─────────────────────────────────────────────
summary_by_group <- map_df(names(METAB_GROUPS), function(g) {
  vars_g <- METAB_GROUPS[[g]]
  res_g  <- metab_res %>% filter(Feature %in% vars_g)
  if (nrow(res_g) == 0) return(NULL)
  data.frame(
    Grupo         = g,
    n_vars        = length(vars_g),
    n_sig_p05     = sum(res_g$p_adj < 0.05, na.rm = TRUE),
    n_sig_p01     = sum(res_g$p_adj < 0.01, na.rm = TRUE),
    mejor_feature = res_g$Feature[which.min(res_g$p_adj)],
    mejor_p_adj   = min(res_g$p_adj, na.rm = TRUE)
  )
})
write.csv(summary_by_group, "Resumen_Significancia_PorGrupo.csv", row.names = FALSE)
cat("✅ Resumen por grupo exportado: Resumen_Significancia_PorGrupo.csv\n")
print(summary_by_group)

# ─── Variables numéricas clínicas ────────────────────────────────────────────
numeric_clinical_cols <- intersect(
  c("age_at_diagnosis.diagnoses", "is_ffpe.samples", "oct_embedded.samples",
    "Prior_Treatment_Flag", "Metastasis_Flag", "ER_PR_HER2_Combo_encoded", "Subtype_encoded"),
  colnames(analysis)
)
clinical_num_res <- run_stats(analysis, numeric_clinical_cols)

# ===============================
# 7. FUNCIONES DE PLOTEO
# ===============================

# --- Violin + Boxplot con color de significancia ---
plot_box_feat <- function(feat, data, stats_df) {
  if (!feat %in% colnames(data)) return(NULL)
  p_info <- stats_df %>% filter(Feature == feat)
  if (nrow(p_info) == 0) return(NULL)
  p_info <- p_info %>% slice(1)

  plot_data <- data %>% filter(is.finite(!!sym(feat)), !is.na(!!sym(feat)))
  if (nrow(plot_data) == 0) return(NULL)

  sig_color <- if (!is.na(p_info$p_adj) && p_info$p_adj < 0.05) "#e63946" else "grey60"
  subtitle_txt <- paste0(
    "p-adj: ", signif(p_info$p_adj, 3),
    if (!is.na(p_info$Fold_Change)) paste0("  |  FC: ", signif(p_info$Fold_Change, 2)) else ""
  )

  ggplot(plot_data, aes(x = Group, y = !!sym(feat), fill = Group)) +
    geom_violin(alpha = 0.2, color = NA) +
    geom_boxplot(width = 0.4, outlier.shape = 16, outlier.size = 2) +
    labs(title = feat, subtitle = subtitle_txt, y = NULL) +
    scale_fill_manual(values = group_colors) +
    base_theme +
    theme(
      legend.position = "none",
      plot.title      = element_text(size = 14, color = sig_color, face = "bold"),
      plot.subtitle   = element_text(size = 12)
    )
}

# --- Heatmap de medianas por grupo ---
plot_heatmap_summary <- function(data, stats_df, group_name = "Todas") {
  if (nrow(stats_df) == 0) return(NULL)

  hm_data <- stats_df %>%
    select(Feature, Med_Core, Med_Div) %>%
    pivot_longer(cols = c(Med_Core, Med_Div), names_to = "Group", values_to = "Median") %>%
    mutate(Group = recode(Group, "Med_Core" = "Core", "Med_Div" = "Divergent")) %>%
    group_by(Feature) %>%
    mutate(Median_scaled = scale(Median)[,1]) %>%
    ungroup()

  ggplot(hm_data, aes(x = Group, y = Feature, fill = Median_scaled)) +
    geom_tile(color = "white") +
    scale_fill_gradient2(low = "#0072B2", mid = "white", high = "#D55E00",
                         midpoint = 0, name = "Mediana\n(z-score)") +
    labs(title = paste("Heatmap:", group_name), x = "", y = "") +
    base_theme +
    theme(axis.text.y = element_text(size = 11))
}

# --- Variables clínicas categóricas/numéricas ---
plot_clinical <- function(var, data) {
  if (!var %in% colnames(data)) return(NULL)
  tmp <- data %>% filter(!is.na(!!sym(var)))
  if (nrow(tmp) == 0) return(NULL)
  tmp[[var]] <- trimws(as.character(tmp[[var]]))

  if (is.numeric(tmp[[var]])) {
    ggplot(tmp, aes(x = Group, y = !!sym(var), fill = Group)) +
      geom_boxplot(outlier.shape = 16, outlier.size = 2) +
      stat_compare_means(label = "p.signif", label.size = 5) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, x = "") +
      base_theme
  } else {
    tbl   <- table(tmp[[var]], tmp$Group)
    p_chi <- tryCatch(chisq.test(tbl)$p.value, error = function(e) NA)
    subtitle_txt <- if (!is.na(p_chi)) paste("Chi² p:", signif(p_chi, 3)) else ""

    ggplot(tmp, aes(x = !!sym(var), fill = Group)) +
      geom_bar(position = "fill") +
      scale_y_continuous(labels = percent) +
      scale_fill_manual(values = group_colors) +
      labs(title = var, subtitle = subtitle_txt, y = "Proporción") +
      base_theme +
      theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 12))
  }
}

# ─── 🔗 Análisis integrado: top metabólicas × variables clínicas clave ──────
# Evalúa si las top metabólicas se asocian con Subtype, Stage, ER/HER2, Metastasis
plot_metab_vs_clinical <- function(metab_feat, clin_var, data) {
  if (!metab_feat %in% colnames(data)) return(NULL)
  if (!clin_var   %in% colnames(data)) return(NULL)

  tmp <- data %>%
    filter(!is.na(!!sym(metab_feat)), !is.na(!!sym(clin_var)),
           is.finite(!!sym(metab_feat))) %>%
    mutate(ClinVar = trimws(as.character(!!sym(clin_var))))

  if (nrow(tmp) == 0 || length(unique(tmp$ClinVar)) < 2) return(NULL)

  ggplot(tmp, aes(x = ClinVar, y = !!sym(metab_feat), fill = Group)) +
    geom_boxplot(outlier.shape = 16, outlier.size = 1.5, alpha = 0.8) +
    scale_fill_manual(values = group_colors) +
    labs(
      title    = paste0(metab_feat, " × ", clin_var),
      subtitle = "¿La diferencia Core/Divergent es independiente del fenotipo clínico?",
      x        = clin_var, y = metab_feat
    ) +
    base_theme +
    theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 10))
}

# ===============================
# 8. GENERACIÓN DEL REPORTE PDF
# ===============================
pdf("Reporte_Metabolismo_Clinico_2026.pdf", width = 14, height = 11)

# ─── PÁGINA DE TÍTULO ────────────────────────────────────────────────────────
grid.newpage()
grid.text("Reporte: Core vs Divergent\nMetabolismo + Clínica + Supervivencia",
          gp = gpar(fontsize = 28, fontface = "bold"), y = 0.72)
grid.text(paste("Pacientes Core:", sum(analysis$Group == "Core"),
                "  |  Divergentes:", sum(analysis$Group == "Divergent")),
          gp = gpar(fontsize = 18), y = 0.58)
grid.text(paste("Variables metabólicas analizadas:", length(metab_available),
                "  |  Significativas (p_adj<0.05):", sum(metab_res$p_adj < 0.05, na.rm = TRUE)),
          gp = gpar(fontsize = 16, col = "grey40"), y = 0.48)
grid.text(paste("Soluciones FBA:", paste(SOLS, collapse = ", ")),
          gp = gpar(fontsize = 14, col = "grey50"), y = 0.38)
grid.text("Estructura: Clínica → Metabolismo → Integración → Supervivencia",
          gp = gpar(fontsize = 14, col = "#0072B2"), y = 0.28)

# ============================================================
# SECCIÓN I: DIFERENCIAS CLÍNICAS
# ============================================================

# ─── Página de sección clínica ───────────────────────────────────────────────
grid.newpage()
grid.text("SECCIÓN I\nDiferencias Clínicas: Core vs Divergent",
          gp = gpar(fontsize = 30, fontface = "bold", col = "#0072B2"), y = 0.60)
grid.text("Macro-categorías: Biología Tumoral | Estadio | Historia Terapéutica | Factores del Paciente",
          gp = gpar(fontsize = 16, col = "grey40"), y = 0.42)

# ─── Iterar por macro-categoría clínica ──────────────────────────────────────
for (clin_gname in names(CLIN_GROUPS)) {
  vars_clin <- CLIN_GROUPS[[clin_gname]]
  if (length(vars_clin) == 0) next

  # Separar numéricas vs categóricas
  numeric_vars <- vars_clin[sapply(vars_clin, function(v) {
    is.numeric(analysis[[v]])
  })]
  categ_vars <- setdiff(vars_clin, numeric_vars)

  # Boxplots para numéricas (de a 6)
  if (length(numeric_vars) > 0) {
    num_plots_clin <- map(numeric_vars, ~plot_box_feat(.x, analysis, clinical_num_res)) %>%
      Filter(Negate(is.null), .)
    if (length(num_plots_clin) > 0) {
      for (i in seq(1, length(num_plots_clin), by = 6)) {
        idx    <- i:min(i + 5, length(num_plots_clin))
        n_cols <- min(3, length(idx))
        n_rows <- ceiling(length(idx) / n_cols)
        grid.arrange(
          grobs = num_plots_clin[idx],
          ncol  = n_cols, nrow = n_rows,
          top   = textGrob(paste("Clínica Numérica —", clin_gname),
                           gp = gpar(fontsize = 18, fontface = "bold"))
        )
      }
    }
  }

  # Barras para categóricas (de a 2)
  if (length(categ_vars) > 0) {
    cat_plots_clin <- map(categ_vars, ~plot_clinical(.x, analysis)) %>%
      Filter(Negate(is.null), .)
    if (length(cat_plots_clin) > 0) {
      for (i in seq(1, length(cat_plots_clin), by = 2)) {
        idx <- i:min(i + 1, length(cat_plots_clin))
        grid.arrange(
          grobs = cat_plots_clin[idx], nrow = length(idx), ncol = 1,
          top   = textGrob(paste("Clínica Categórica —", clin_gname),
                           gp = gpar(fontsize = 18, fontface = "bold"))
        )
      }
    }
  }
}

# ============================================================
# SECCIÓN II: DIFERENCIAS METABÓLICAS
# ============================================================
grid.newpage()
grid.text("SECCIÓN II\nDiferencias Metabólicas: Core vs Divergent",
          gp = gpar(fontsize = 30, fontface = "bold", col = "#D55E00"), y = 0.60)
grid.text("5 bloques funcionales: Energía | Redox | Lípidos | Dependencias | Reprogramación Sistémica",
          gp = gpar(fontsize = 16, col = "grey40"), y = 0.42)

# ─── Heatmap resumen global (top 40 por p_adj) ───────────────────────────────
if (nrow(metab_res) > 0) {
  metab_res_hm <- metab_res %>%
    filter(!is.na(p_adj)) %>%
    arrange(p_adj) %>%
    head(40)
  hm <- plot_heatmap_summary(analysis, metab_res_hm, "Top 40 variables metabólicas (por p_adj)")
  if (!is.null(hm)) print(hm)
}

# ─── Sección por bloque metabólico funcional ─────────────────────────────────
for (group_name in names(METAB_GROUPS)) {
  vars_in_group <- METAB_GROUPS[[group_name]]
  if (length(vars_in_group) == 0) next

  stats_group <- metab_res %>% filter(Feature %in% vars_in_group)
  if (nrow(stats_group) == 0) next

  # Heatmap del bloque (máx 40)
  stats_hm <- if (nrow(stats_group) > 40) head(stats_group, 40) else stats_group
  hm_g <- plot_heatmap_summary(analysis, stats_hm, group_name)
  if (!is.null(hm_g)) print(hm_g)

  # Violin plots de a 6
  plots_g <- map(vars_in_group, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)

  if (length(plots_g) > 0) {
    for (i in seq(1, length(plots_g), by = 6)) {
      idx    <- i:min(i + 5, length(plots_g))
      n_cols <- min(3, length(idx))
      n_rows <- ceiling(length(idx) / n_cols)
      grid.arrange(
        grobs = plots_g[idx],
        ncol  = n_cols, nrow = n_rows,
        top   = textGrob(paste("Metabolismo —", group_name),
                         gp = gpar(fontsize = 18, fontface = "bold"))
      )
    }
  }
}

# ─── Top 12 metabólicas: p_adj + efecto (log2FC) ─────────────────────────────
# ✅ MEJORA: prioriza FDR + tamaño de efecto
top12 <- metab_res %>%
  filter(!is.na(p_adj), !is.na(Fold_Change), Fold_Change > 0) %>%
  arrange(p_adj, desc(abs(log2(Fold_Change)))) %>%
  head(12)

cat("\n🏆 Top 12 metabólicas (p_adj + log2FC):\n")
print(top12 %>% select(Feature, p_adj, Fold_Change))

if (nrow(top12) >= 2) {
  top12_plots <- map(top12$Feature, ~plot_box_feat(.x, analysis, metab_res)) %>%
    Filter(Negate(is.null), .)
  if (length(top12_plots) > 0) {
    for (i in seq(1, length(top12_plots), by = 6)) {
      idx    <- i:min(i + 5, length(top12_plots))
      n_cols <- min(3, length(idx))
      n_rows <- ceiling(length(idx) / n_cols)
      grid.arrange(
        grobs = top12_plots[idx],
        ncol  = n_cols, nrow = n_rows,
        top   = textGrob("🏆 Top 12 Variables Metabólicas (p_adj + efecto)",
                         gp = gpar(fontsize = 20, fontface = "bold"))
      )
    }
  }
}

# ============================================================
# SECCIÓN III: ANÁLISIS INTEGRADO CLÍNICA ↔ METABOLISMO
# ============================================================
grid.newpage()
grid.text("SECCIÓN III\nAnálisis Integrado: Clínica ↔ Metabolismo",
          gp = gpar(fontsize = 30, fontface = "bold", col = "#009E73"), y = 0.60)
grid.text("¿Core vs Divergent es reflejo del subtipo? ¿O reprogramación metabólica independiente?",
          gp = gpar(fontsize = 16, col = "grey40"), y = 0.42)

# Variables clínicas ancla para integración
integration_clin_vars <- intersect(
  c("Subtype", "ajcc_pathologic_stage.diagnoses", "ER", "HER2", "Metastasis_Flag"),
  colnames(analysis)
)

top_features <- top12$Feature

# Para cada variable clínica ancla, mostrar top 3 metabólicas
for (clin_var in integration_clin_vars) {
  integ_plots <- map(head(top_features, 3), ~plot_metab_vs_clinical(.x, clin_var, analysis)) %>%
    Filter(Negate(is.null), .)

  if (length(integ_plots) > 0) {
    grid.arrange(
      grobs = integ_plots,
      ncol  = min(3, length(integ_plots)),
      nrow  = 1,
      top   = textGrob(
        paste0("Integración: Top metabólicas × ", clin_var),
        gp = gpar(fontsize = 18, fontface = "bold", col = "#009E73")
      )
    )
  }
}

# ─── Heatmap integrado: top metabólicas × subtype ───────────────────────────
# Muestra medianas de top metabólicas estratificadas por Subtype y Group
if ("Subtype" %in% colnames(analysis) && length(top_features) > 0) {
  subtype_metab <- analysis %>%
    filter(!is.na(Subtype)) %>%
    select(Group, Subtype, all_of(intersect(top_features, colnames(analysis)))) %>%
    pivot_longer(cols = -c(Group, Subtype), names_to = "Feature", values_to = "Value") %>%
    group_by(Group, Subtype, Feature) %>%
    summarise(Median = median(Value, na.rm = TRUE), .groups = "drop") %>%
    group_by(Feature) %>%
    mutate(Median_z = scale(Median)[,1]) %>%
    ungroup() %>%
    mutate(GroupSubtype = paste0(Group, "\n", Subtype))

  if (nrow(subtype_metab) > 0) {
    print(
      ggplot(subtype_metab, aes(x = GroupSubtype, y = Feature, fill = Median_z)) +
        geom_tile(color = "white") +
        scale_fill_gradient2(low = "#0072B2", mid = "white", high = "#D55E00",
                             midpoint = 0, name = "Mediana\n(z-score)") +
        labs(
          title    = "Heatmap Integrado: Top Metabólicas × Subtype × Grupo",
          subtitle = "Si el patrón Core/Divergent persiste dentro de cada subtipo → efecto independiente",
          x = "", y = ""
        ) +
        base_theme +
        theme(axis.text.x = element_text(angle = 40, hjust = 1, size = 12),
              axis.text.y = element_text(size = 11))
    )
  }
}

# ============================================================
# SECCIÓN IV: KAPLAN-MEIER
# ============================================================
grid.newpage()
grid.text("SECCIÓN IV\nAnálisis de Supervivencia",
          gp = gpar(fontsize = 30, fontface = "bold", col = "#CC79A7"), y = 0.60)
grid.text("¿El fenotipo metabólico Core/Divergent predice pronóstico?",
          gp = gpar(fontsize = 16, col = "grey40"), y = 0.42)

if (all(c("OS.time", "OS") %in% colnames(analysis))) {
  km_data <- analysis %>%
    filter(!is.na(OS.time), !is.na(OS), OS.time > 0)

  if (nrow(km_data) > 0 && length(unique(km_data$Group)) == 2) {
    fit <- survfit(Surv(OS.time, OS) ~ Group, data = km_data)
    print(ggsurvplot(fit, data = km_data,
                     pval           = TRUE,
                     pval.size      = 5,
                     conf.int       = TRUE,
                     palette        = unname(group_colors),
                     risk.table     = TRUE,
                     risk.table.cex = 1.0,
                     surv.line.size = 1.2,
                     title          = "Curva de Supervivencia: Core vs Divergent",
                     xlab           = "Tiempo (días)",
                     ylab           = "Probabilidad de Supervivencia",
                     legend.labs    = c("Core", "Divergent"),
                     theme          = base_theme +
                       theme(plot.title  = element_text(size = 24, face = "bold"),
                             axis.title  = element_text(size = 18),
                             axis.text   = element_text(size = 16),
                             legend.text = element_text(size = 16))))
  } else {
    warning("⚠️  Datos insuficientes para Kaplan-Meier tras filtrado.")
  }
} else {
  warning("⚠️  Columnas OS y/o OS.time no encontradas en el dataset metabólico.")
}

dev.off()

cat("\n✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'\n")
cat("✅ Tabla estadística: 'Estadisticas_Metabolicas_CoreVsDivergent.csv'\n")
cat("✅ Resumen por grupo: 'Resumen_Significancia_PorGrupo.csv'\n")
cat("\n📌 Estructura del reporte:\n")
cat("   Sección I   — Diferencias Clínicas (4 macro-categorías)\n")
cat("   Sección II  — Diferencias Metabólicas (5 bloques funcionales + Top 12)\n")
cat("   Sección III — Análisis Integrado Clínica ↔ Metabolismo\n")
cat("   Sección IV  — Kaplan-Meier\n")


🔍 Subsistemas SA detectados en CSV: 201 
   Subsistemas únicos: 85 
   • SA_Alanine_and_aspartate_metabolism
   • SA_Aminosugar_metabolism
   • SA_Androgen_and_estrogen_synthesis_and_metabolism
   • SA_Arachidonic_acid_metabolism
   • SA_Arginine_and_proline_metabolism
   • SA_Beta_Alanine_metabolism
   • SA_Bile_acid_synthesis
   • SA_Biomass_reactions
   • SA_Biotin_metabolism
   • SA_Butanoate_metabolism 
   ... 75 más

📊 Variables metabólicas encontradas: 248 / 286
⚠️  Variables NO encontradas ( 38 ):
   • LipidSat_pFBA
   • LipidUnsat_pFBA
   • CU_L2
   • EA_L2
   • WarburgIndex_L2
   • ATPConsumption_L2
   • ATPProduction_L2
   • RedoxIndex_L2
   • MFI_L2
   • AnabolismScore_L2
   • NADPHdemand_L2
   • TCA_completeness_L2
   • LipidSat_L2
   • LipidUnsat_L2
   • LipidPL_L2
   • GlnDependence_L2
   • CU_L2w
   • EA_L2w
   • WarburgIndex_L2w
   • ATPConsumption_L2w 
   ... 18 más

✅ Pacientes Core: 1122  | Divergent: 44 

📂 Grupos clínicos activos: 4 
   • 🧬 A. Biología Tumoral : 

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Estructura: Clínica → Metabolismo → Integración → Supervivencia' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation may be incorrect”
Warning message in chisq.test(tbl):
“Chi-squared approximation m


🏆 Top 12 metabólicas (p_adj + log2FC):
                            Feature        p_adj Fold_Change
1          SA_Biomass_reactions_FBA 5.867181e-90   1.0000000
2          SA_Biomass_reactions_L1w 5.867181e-90   1.0000000
3         SA_Biomass_reactions_pFBA 5.867181e-90   1.0000000
4                GlnDependence_pFBA 5.901350e-57   1.0000000
5                      LipidPL_pFBA 5.371994e-55   1.0000000
6            SA_Drug_metabolism_L1w 4.076916e-06   0.1297907
7  SA_Oxidative_phosphorylation_L1w 2.201802e-05   0.6632516
8         SA_Purine_catabolism_pFBA 2.227023e-05   0.6111772
9                ATPConsumption_L1w 2.227023e-05   0.7697850
10                ATPProduction_L1w 2.227023e-05   0.7697850
11                           CU_L1w 4.579437e-05   0.9644650
12         SA_Purine_catabolism_L1w 8.938182e-05   0.7261486


Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on '🏆 Top 12 Variables Metabólicas (p_adj + efecto)' in 'mbcsToSbcs': for 🏆 (U+1F3C6)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on '🏆 Top 12 Variables Metabólicas (p_adj + efecto)' in 'mbcsToSbcs': for 🏆 (U+1F3C6)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Análisis Integrado: Clínica ↔ Metabolismo' in 'mbcsToSbcs': <-> substituted for ↔ (U+2194)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'Si el patrón Core/Divergent persiste dentro de cada subtipo → efecto independiente' in 'mbcsToSbcs': -> substituted for → (U+2192)”
Ignoring unknown labels:
• colour : "Strata"


pdf 
  2


✅ Reporte generado: 'Reporte_Metabolismo_Clinico_2026.pdf'
✅ Tabla estadística: 'Estadisticas_Metabolicas_CoreVsDivergent.csv'
✅ Resumen por grupo: 'Resumen_Significancia_PorGrupo.csv'

📌 Estructura del reporte:
   Sección I   — Diferencias Clínicas (4 macro-categorías)
   Sección II  — Diferencias Metabólicas (5 bloques funcionales + Top 12)
   Sección III — Análisis Integrado Clínica ↔ Metabolismo
   Sección IV  — Kaplan-Meier


In [8]:
install.packages("pheatmap")

Installing package into ‘/Users/eduardoruiz/Library/R/arm64/4.4/library’
(as ‘lib’ is unspecified)




The downloaded binary packages are in
	/var/folders/x0/j78w4f8n0_z4l575_k16n2k40000gn/T//RtmpPiHpQZ/downloaded_packages
